# 📊 Trabajo Práctico – Sistemas Expertos
---
## Materia: Desarrollo de Sistemas de IA
### Grupo 8
### Integrantes:
- Leandro Marinovich (Coordinación-Revisión-Documentación)
- Pablo Molinari (Generación de Código y Reglas)
- Valentín Roldán (Representación de Frames-Encargado de Demostración Práctica)
- José Sabia (Carga de datos-Análisis de Conclusiones)


## 1️⃣ Introducción al Proyecto


Hemos decidido trabajar con el dataset elegido en la anterior etapa de análisis EDA. Allí, se trabaja con un conjunto de datos relacionado a las obras públicas dentro de Argentina. Todo es información sobre el presupuesto y la ejecución de las obras del Ministerio de Obras Públicas de la Repúblia Argentina.

En esta parte, decidimos ya utilizar el dataset con los datos limpios, guardados en un nuevo csv, con el fin de ya trabajar directamente con lo solicitado en Sistemas Expertos.

El problema central que este Sistema Experto (SE) resuelve es la gestión de riesgos reactiva y la dependencia del criterio humano en el monitoreo de miles de proyectos de Obras Públicas a nivel provincial. Actualmente, los gestores luchan por transformar un gran volumen de datos brutos en diagnósticos accionables, lo que permite que anomalías críticas como la sobrecertificación de obras (donde el pago supera ampliamente el avance físico) o la inactividad prolongada de proyectos pasen desapercibidas hasta que el daño financiero o social es irreparable. Este sistema es vital porque la toma de decisiones, como ordenar una auditoría o reasignar fondos, se vuelve lenta e inconsistente.

El SE ofrece una solución al automatizar el razonamiento experto, convirtiéndose en un analista de riesgo y eficiencia las 24 horas del día. Su objetivo es proporcionar un soporte a la decisión proactivo a gestores provinciales y ministerios nacionales. Esto se logra mediante un Motor de Inferencia que aplica reglas heurísticas codificadas, como la Regla de Desequilibrio Financiero (si Avance Financiero ≥80% y Avance Físico <50%).

Al seleccionar una provincia, el sistema entrega un diagnóstico claro ("Alto Riesgo de Desequilibrio Financiero") con una recomendación de acción ("Auditar proyectos con mayor brecha"), haciendo que la gestión del riesgo sea inmediata, objetiva y basada en evidencia.

A continuación, cargamos el dataframe y hacemos un resumen de sus características principales, columnas y estadísticas relevantes.

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

ValueError: mount failed

In [ ]:
inversiones_df = pd.read_csv('/content/drive/MyDrive/Obras_Publicas_Limpio.csv')


In [ ]:
inversiones_df.head()

,id_proyecto,numero_obra,anio_inicio,anio_fin,nombre_obra,descripcion_fisica,monto_total,sector,avance_financiero,avance_fisico,entidad_ejecutora,duracion_dias,objetivo_general,tipo_proyecto,departamento,provincia,etapa_obra,tipo_moneda
0,1610,NA70056,2019,2020,CIERRE DE MALLA RINCON DE MILBERG – MODULO 3,Instalación de red secundaria de agua en Sec...,17872007.0,AGUA Y CLOACA,100.0,100.0,AGUA Y SANEAMIENTOS ARGENTINOS,443,Instalación de red secundaria de agua en Sec...,RED SECUNDARIA,TIGRE,BUENOS AIRES,FINALIZADAS,pesos argentinos
1,1613,NA70116,2019,2021,RED SECUNDARIA DE AGUA B EL ARCO M2 + AMPLIACION,Instalación por vereda de cañería de DN 160 mm...,7428947.0,AGUA Y CLOACA,100.0,100.0,AGUA Y SANEAMIENTOS ARGENTINOS,471,Instalación por vereda de cañería de DN 160 mm...,RED SECUNDARIA,TIGRE,BUENOS AIRES,FINALIZADAS,pesos argentinos
2,1614,NA70152,2019,2021,CIERRE DE MALLA RINCON DE MILBERG – MODULO 2,El objetivo de la obra es la ejecución de una ...,6390504.0,AGUA Y CLOACA,100.0,100.0,AGUA Y SANEAMIENTOS ARGENTINOS,831,El objetivo de la obra es la ejecución de una ...,RED SECUNDARIA,TIGRE,BUENOS AIRES,FINALIZADAS,pesos argentinos
3,1615,NA70184,2019,2021,RED SECUNDARIA DE AGUA B PERUZOTTI MOD 1,El objetivo de la obra es la ejecución de una ...,11193140.0,AGUA Y CLOACA,100.0,100.0,AGUA Y SANEAMIENTOS ARGENTINOS,691,El objetivo de la obra es la ejecución de una ...,RED SECUNDARIA,PILAR,BUENOS AIRES,FINALIZADAS,pesos argentinos
4,1617,NA70196,2022,2023,RED SECUNDARIA DE AGUA B PERUZOTTI MOD 2,El objetivo de la obra es la ejecución de una ...,38500317.0,AGUA Y CLOACA,100.0,78.0,AGUA Y SANEAMIENTOS ARGENTINOS,330,El objetivo de la obra es la ejecución de una ...,RED SECUNDARIA,PILAR,BUENOS AIRES,EN EJECUCIÓN,pesos argentinos


In [ ]:
inversiones_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7285 entries, 0 to 7284
Data columns (total 18 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   id_proyecto         7285 non-null   int64  
 1   numero_obra         7285 non-null   object 
 2   anio_inicio         7285 non-null   int64  
 3   anio_fin            7285 non-null   int64  
 4   nombre_obra         7285 non-null   object 
 5   descripcion_fisica  7285 non-null   object 
 6   monto_total         7285 non-null   float64
 7   sector              7285 non-null   object 
 8   avance_financiero   7285 non-null   float64
 9   avance_fisico       7285 non-null   float64
 10  entidad_ejecutora   7285 non-null   object 
 11  duracion_dias       7285 non-null   int64  
 12  objetivo_general    7285 non-null   object 
 13  tipo_proyecto       7285 non-null   object 
 14  departamento        7285 non-null   object 
 15  provincia           7285 non-null   object 
 16  etapa_

In [ ]:
inversiones_df.describe().T

,count,mean,std,min,25%,50%,75%,max
id_proyecto,7285.0,7.241969e+08,4.483412e+08,1610.0,10014463.00,1.003112e+09,1.003119e+09,1.003131e+09
anio_inicio,7285.0,2.021288e+03,1.609048e+00,2008.0,2021.00,2.021000e+03,2.022000e+03,2.024000e+03
anio_fin,7285.0,2.022369e+03,1.339951e+00,2020.0,2021.00,2.022000e+03,2.023000e+03,2.029000e+03
monto_total,7285.0,8.703218e+08,6.050980e+09,18.0,11573078.00,3.571195e+07,1.573665e+08,2.179916e+11
avance_financiero,7285.0,7.260367e+01,3.368690e+01,0.0,46.32,9.026000e+01,1.000000e+02,1.000000e+02
avance_fisico,7285.0,7.927491e+01,3.237015e+01,0.0,63.91,1.000000e+02,1.000000e+02,1.000000e+02
duracion_dias,7285.0,3.757218e+02,5.207669e+02,1.0,120.00,1.810000e+02,4.160000e+02,6.454000e+03


## 2️⃣ Representación del Conocimiento

### 2.1 Marcos (Frames)

In [ ]:
# EJEMPLO DE FRAME: Obra de Saneamiento Finalizada (ID 1610)

proyecto_finalizado_1610 = {
    'tipo_entidad': 'Proyecto de Infraestructura',

    #Identificación
    'id_proyecto': 1610,
    'numero_obra': 'NA70056',
    'nombre_obra': 'CIERRE DE MALLA RINCON DE MILBERG – MODULO 3',

    # Ubicación
    'provincia': 'BUENOS AIRES',
    'departamento': 'TIGRE',
    'sector': 'AGUA Y CLOACA',
    'tipo_proyecto': 'RED SECUNDARIA',

    # Estado
    'etapa_obra': 'FINALIZADAS',
    'anio_inicio': 2019,
    'anio_fin': 2020,
    'duracion_dias': 443,
    'monto_total': 17872007.0,

    # Avance
    'avance_fisico': 100.0,
    'avance_financiero': 100.0,

    # Entidad Ejecutora
    'entidad_ejecutora': 'AGUA Y SANEAMIENTOS ARGENTINOS'
}

print(proyecto_finalizado_1610)

{'tipo_entidad': 'Proyecto de Infraestructura', 'id_proyecto': 1610, 'numero_obra': 'NA70056', 'nombre_obra': 'CIERRE DE MALLA RINCON DE MILBERG – MODULO 3', 'provincia': 'BUENOS AIRES', 'departamento': 'TIGRE', 'sector': 'AGUA Y CLOACA', 'tipo_proyecto': 'RED SECUNDARIA', 'etapa_obra': 'FINALIZADAS', 'anio_inicio': 2019, 'anio_fin': 2020, 'duracion_dias': 443, 'monto_total': 17872007.0, 'avance_fisico': 100.0, 'avance_financiero': 100.0, 'entidad_ejecutora': 'AGUA Y SANEAMIENTOS ARGENTINOS'}


### 2.2 Reglas SI-ENTONCES: Condiciones Lógicas para Obras Públicas



Estas condiciones son las premisas que el **Motor de Inferencia** evaluará sobre el *DataFrame* filtrado por la provincia seleccionada, transformando los datos en diagnósticos de riesgo y eficiencia.


#### **Regla 1: Alerta de Inactividad Crítica**

Esta condición identifica proyectos que han vencido (su **`anio_fin`** es anterior al año actual) y no muestran avance, permitiendo al sistema recomendar la reasignación de fondos.

$$\textbf{SI } (\%\text{ de Proyectos con } \mathbf{avance\_fisico} < 5\%) \textbf{ Y } (\mathbf{anio\_fin} < \text{Año Actual}) \textbf{ Y } (\text{Total de proyectos en esta condición} \ge 0.05$$ Total de proyectos de provincia)

---
#### **Regla 2: Riesgo de Desequilibrio Financiero (Sobrecertificación)**

Esta regla se activa si el flujo de caja está desproporcionadamente alto respecto al progreso físico, señalando un posible riesgo financiero o una auditoría necesaria.

$$\textbf{SI } (\text{Promedio } \mathbf{avance\_financiero} \ge 80\%) \textbf{ Y } (\text{Promedio } \mathbf{avance\_fisico} < 50\%)$$

---

#### **Regla 3: Clasificación de Eficiencia en Duración**

Esta regla clasifica positivamente a la provincia cuya gestión de proyectos es excepcionalmente rápida en comparación con el promedio nacional, sugiriendo un modelo de éxito a replicar.

$$\textbf{SI } (\text{Promedio } \mathbf{duracion\_dias} \text{ Provincial} < \text{Media Nacional de } \mathbf{duracion\_dias} - 10\%)$$

---

#### **Regla 4: Alerta de Fragmentación de Inversión**

Esta condición se activa cuando el **`monto_total`** promedio de las obras es bajo, pero el *número* de proyectos es alto, diagnosticando una inversión fragmentada que puede mermar el impacto y la eficiencia administrativa.

$$\textbf{SI } (\text{Promedio } \mathbf{monto\_total} \text{ Provincial} < \text{Media Nacional de } \mathbf{monto\_total} - 30\%) \textbf{ Y } (\text{Número de Proyectos} > \text{Media Nacional de Proyectos})$$

## 3️⃣ Motor de Inferencia

**PRIMERA REGLA: Riesgo de Inactividad**

In [ ]:
def diagnostico_riesgo_inactividad(df: pd.DataFrame, provincia_seleccionada: str, umbral_riesgo_porcentual=0.05):

    # Generamos df de la provincia
    df_provincia = df[df['provincia'] == provincia_seleccionada].copy()

    if df_provincia.empty:
        return {
            'Provincia': provincia_seleccionada,
            'Conclusion': 'ERROR: No se encontraron datos.',
            'Recomendacion': 'Verifique el nombre de la provincia.'
        }

    # PASO 1: Calcular totales y determinar el año de corte
    ANIO_ACTUAL = datetime.now().year
    TOTAL_PROYECTOS_PROVINCIAL = len(df_provincia)

    # PASO 2: Filtrar los "Hechos" que cumplen la condición SI

    # Condición A: Avance Físico es menor al 5% (0.05)
    proyectos_baja_ejecucion = df_provincia[df_provincia['avance_fisico'] < 0.05]

    # Condición B: Año de Finalización es anterior al año actual
    proyectos_zombie = proyectos_baja_ejecucion[
        proyectos_baja_ejecucion['anio_fin'] < ANIO_ACTUAL
    ]

    # Conteo de proyectos que cumplen ambas condiciones
    conteo_zombie = len(proyectos_zombie)

    # PASO 3: Evaluar la Condición Final y Generar la Conclusión (ENTONCES)

    # El umbral ahora se calcula dinámicamente:
    UMBRAL_PROYECTOS_RIESGO = TOTAL_PROYECTOS_PROVINCIAL * umbral_riesgo_porcentual

    if conteo_zombie >= UMBRAL_PROYECTOS_RIESGO:

        porcentaje_zombie = (conteo_zombie / TOTAL_PROYECTOS_PROVINCIAL) * 100

        return {
            'Provincia': provincia_seleccionada,
            'Regla': 'R2 (Inactividad Crítica) - ACTIVADA',
            'Conclusion': f"ALERTA CRÍTICA: Se detectó que el {porcentaje_zombie:.1f}% de la cartera ({conteo_zombie} proyectos) está en condición 'zombie'.",
            'Recomendacion': 'Evaluar urgentemente la cancelación o reasignación de fondos de estas obras para liberar recursos y reducir la carga administrativa.'
        }
    else:
        # Resultado si la regla NO se activa
        porcentaje_zombie = (conteo_zombie / TOTAL_PROYECTOS_PROVINCIAL) * 100 if TOTAL_PROYECTOS_PROVINCIAL > 0 else 0

        return {
            'Provincia': provincia_seleccionada,
            'Regla': 'R2 (Inactividad Crítica) - DESACTIVADA',
            'Conclusion': f"Riesgo bajo. El {porcentaje_zombie:.1f}% de la cartera ({conteo_zombie} proyectos) está inactivo, por debajo del umbral del {umbral_riesgo_porcentual*100}%.",
            'Recomendacion': 'Mantener el monitoreo de avance físico y financiero.'
        }

**SEGUNDA REGLA: Diagnóstico de desequilibrio financiero**

In [ ]:
def diagnostico_desequilibrio_financiero(df: pd.DataFrame, provincia_seleccionada: str):

    df_provincia = df[df['provincia'] == provincia_seleccionada].copy()

    if df_provincia.empty:
        return {'Provincia': provincia_seleccionada, 'Conclusion': 'ERROR: No se encontraron datos.'}

    # PASO 1: Calcular los hechos (promedios) de la provincia
    promedio_avance_financiero = df_provincia['avance_financiero'].mean() / 100
    promedio_avance_fisico = df_provincia['avance_fisico'].mean() / 100

    # PASO 2: Evaluar la Condición (SI)
    # SI (Avance Financiero >= 80%) Y (Avance Físico < 50%)
    if (promedio_avance_financiero >= 0.80) and (promedio_avance_fisico < 0.50):
        # PASO 3: Generar la Conclusión (ENTONCES)
        return {
            'Provincia': provincia_seleccionada,
            'Regla': 'R2 (Desequilibrio Financiero)',
            'Promedio_Financiero': promedio_avance_financiero,
            'Promedio_Fisico': promedio_avance_fisico,
            'Conclusion': 'ALERTA: Alto Riesgo de Desequilibrio Financiero. Sobrecertificación potencial.',
            'Recomendacion': 'Auditar los certificados de obra y pagos para verificar la correcta ejecución física.'
        }
    else:
        return {
            'Provincia': provincia_seleccionada,
            'Promedio_Financiero': promedio_avance_financiero,
            'Promedio_Fisico': promedio_avance_fisico,
            'Conclusion': 'No se detecta un desequilibrio crítico entre avance financiero y físico.',
            'Recomendacion': 'Continuar con el monitoreo normal de ejecución.'
        }


**TERCERA REGLA: Eficiencia en Duración**

In [ ]:
def diagnostico_eficiencia_duracion(df: pd.DataFrame, provincia_seleccionada: str, MEDIA_DURACION_NACIONAL: float):

    df_provincia = df[df['provincia'] == provincia_seleccionada].copy()

    if df_provincia.empty:
        return {'Provincia': provincia_seleccionada, 'Conclusion': 'ERROR: No se encontraron datos.'}

    # PASO 1: Calcular el hecho (promedio de duración) de la provincia
    avg_duracion_provincial = df_provincia['duracion_dias'].mean()

    # PASO 2: Evaluar la Condición
    UMBRAL_EFICIENCIA = 0.90 # 90% de la media nacional

    if avg_duracion_provincial < (MEDIA_DURACION_NACIONAL * UMBRAL_EFICIENCIA):

        # PASO 3: Generar la Conclusión (ENTONCES)
        return {
            'Provincia': provincia_seleccionada,
            'Regla': 'R3 (Eficiencia en Duración)',
            'Conclusion': 'CLASIFICACIÓN: Liderazgo en Eficiencia. Ejecución de obras significativamente más rápida.',
            'Recomendacion': 'Documentar y replicar las mejores prácticas de planificación y gestión logística de la provincia.'
        }
    else:
        return {
            'Provincia': provincia_seleccionada,
            'Conclusion': 'El tiempo promedio de ejecución de obras es comparable a la media nacional.',
            'Recomendacion': 'Buscar oportunidades para optimizar la duración de los proyectos.'
        }

**CUARTA REGLA: Alerta de Fragmentación de Inversión**

In [ ]:
def diagnostico_fragmentacion_inversion(df: pd.DataFrame, provincia_seleccionada: str, MEDIA_MONTO_NACIONAL: float, MEDIA_PROYECTOS_NACIONAL: float) -> dict:

    df_provincia = df[df['provincia'] == provincia_seleccionada].copy()

    if df_provincia.empty:
        return {'Provincia': provincia_seleccionada, 'Conclusion': 'ERROR: No se encontraron datos.'}

    # PASO 1: Calcular los hechos de la provincia
    avg_monto_provincial = df_provincia['monto_total'].mean()
    conteo_proyectos_provincial = len(df_provincia)

    # PASO 2: Evaluar la Condición (SI)
    UMBRAL_FRAGMENTACION_MONTO = 0.70 # 70% de la media nacional

    if (avg_monto_provincial < (MEDIA_MONTO_NACIONAL * UMBRAL_FRAGMENTACION_MONTO)) and \
       (conteo_proyectos_provincial > MEDIA_PROYECTOS_NACIONAL):

        # PASO 3: Generar la Conclusión (ENTONCES)
        return {
            'Provincia': provincia_seleccionada,
            'Regla': 'R4 (Fragmentación de Inversión)',
            'Conclusion': 'ALERTA: Inversión altamente fragmentada en proyectos de bajo monto. Posible ineficiencia administrativa.',
            'Recomendacion': 'Evaluar la consolidación de proyectos futuros para aumentar el impacto económico y reducir la carga de gestión.'
        }
    else:
        return {
            'Provincia': provincia_seleccionada,
            'Conclusion': 'La cartera de proyectos tiene un tamaño y monto promedio saludable.',
            'Recomendacion': 'Mantener el equilibrio entre grandes proyectos y micro-obras de alto impacto social.'
        }

Basado en el trabajo que hemos realizado, el Sistema Experto de Monitoreo de Obras Públicas utiliza el Encadenamiento Hacia Adelante (Forward Chaining) como estrategia de inferencia principal. Este enfoque es la elección correcta porque el sistema no busca confirmar una meta predefinida, sino que toma los datos crudos de la provincia (los "hechos", como el promedio de avance financiero o el conteo de proyectos) y aplica las reglas SI-ENTONCES para descubrir y generar un diagnóstico o una "alerta" de riesgo. Al iniciar el proceso desde los datos hacia la conclusión, se simula un sistema de monitoreo que automáticamente detecta y clasifica problemas como el Desequilibrio Financiero (R2), Proyectos "Zombie" (R1) o Fragmentación de Inversión (R4).

## 4️⃣ Demostración Práctica

**Probamos la Regla 1 con provincias como Buenos Aires y Santiago del Estero.**

In [ ]:
print("--- Diagnóstico para BUENOS AIRES ---")
diagnostico_ba = diagnostico_riesgo_inactividad(inversiones_df, 'BUENOS AIRES')
print(f"Conclusión: {diagnostico_ba['Conclusion']}")
print(f"Recomendación: {diagnostico_ba['Recomendacion']}")

--- Diagnóstico para BUENOS AIRES ---
Conclusión: Riesgo bajo. El 2.0% de la cartera (65 proyectos) está inactivo, por debajo del umbral del 5.0%.
Recomendación: Mantener el monitoreo de avance físico y financiero.


In [ ]:
print("--- Diagnóstico para SANTIAGO DEL ESTERO ---")
diagnostico_sgo = diagnostico_riesgo_inactividad(inversiones_df, 'SANTIAGO DEL ESTERO')
print(f"Conclusión: {diagnostico_sgo['Conclusion']}")
print(f"Recomendación: {diagnostico_sgo['Recomendacion']}")

--- Diagnóstico para SANTIAGO DEL ESTERO ---
Conclusión: ALERTA CRÍTICA: Se detectó que el 14.0% de la cartera (16 proyectos) está en condición 'zombie'.
Recomendación: Evaluar urgentemente la cancelación o reasignación de fondos de estas obras para liberar recursos y reducir la carga administrativa.


**Probamos la Regla 2 con provincias como Córdoba y Chaco.**

In [ ]:
print("--- Diagnóstico para CÓRDOBA ---")
diagnostico_cordoba = diagnostico_desequilibrio_financiero(inversiones_df, 'CÓRDOBA')
print(f"Conclusión: {diagnostico_cordoba['Conclusion']}")
print(f"Recomendación: {diagnostico_cordoba['Recomendacion']}")

--- Diagnóstico para CÓRDOBA ---
Conclusión: No se detecta un desequilibrio crítico entre avance financiero y físico.
Recomendación: Continuar con el monitoreo normal de ejecución.


In [ ]:
print("--- Diagnóstico para CHACO ---")
diagnostico_chaco = diagnostico_desequilibrio_financiero(inversiones_df, 'CHACO')
print(f"Conclusión: {diagnostico_chaco['Conclusion']}")
print(f"Recomendación: {diagnostico_chaco['Recomendacion']}")

--- Diagnóstico para CHACO ---
Conclusión: No se detecta un desequilibrio crítico entre avance financiero y físico.
Recomendación: Continuar con el monitoreo normal de ejecución.


****Probamos la Regla 3 con provincias como Santa Cruz y Santa Fe.****

In [ ]:
#Métrica necesaria para llevar adelante la regla
MEDIA_DURACION_NACIONAL = inversiones_df['duracion_dias'].mean()

In [ ]:
print("--- Diagnóstico para SANTA CRUZ ---")
diagnostico_scz = diagnostico_eficiencia_duracion(inversiones_df, 'SANTA CRUZ', MEDIA_DURACION_NACIONAL)
print(f"Conclusión: {diagnostico_scz['Conclusion']}")
print(f"Recomendación: {diagnostico_scz['Recomendacion']}")

--- Diagnóstico para SANTA CRUZ ---
Conclusión: CLASIFICACIÓN: Liderazgo en Eficiencia. Ejecución de obras significativamente más rápida.
Recomendación: Documentar y replicar las mejores prácticas de planificación y gestión logística de la provincia.


In [ ]:
print("--- Diagnóstico para SANTA FE ---")
diagnostico_sfe = diagnostico_eficiencia_duracion(inversiones_df, 'SANTA FE', MEDIA_DURACION_NACIONAL)
print(f"Conclusión: {diagnostico_sfe['Conclusion']}")
print(f"Recomendación: {diagnostico_sfe['Recomendacion']}")

--- Diagnóstico para SANTA FE ---
Conclusión: El tiempo promedio de ejecución de obras es comparable a la media nacional.
Recomendación: Buscar oportunidades para optimizar la duración de los proyectos.


****Probamos la Regla 4 con provincias como Córdoba y Chaco.****

In [ ]:
#Métricas necesarias para llevar adelante la regla
MEDIA_MONTO_NACIONAL = inversiones_df['monto_total'].mean()
MEDIA_PROYECTOS_NACIONAL = inversiones_df.groupby('provincia')['id_proyecto'].count().mean()

In [ ]:
print("--- Diagnóstico para MENDOZA ---")
diagnostico_mza = diagnostico_fragmentacion_inversion(inversiones_df, 'MENDOZA', MEDIA_MONTO_NACIONAL, MEDIA_PROYECTOS_NACIONAL)
print(f"Conclusión: {diagnostico_mza['Conclusion']}")
print(f"Recomendación: {diagnostico_mza['Recomendacion']}")

--- Diagnóstico para MENDOZA ---
Conclusión: La cartera de proyectos tiene un tamaño y monto promedio saludable.
Recomendación: Mantener el equilibrio entre grandes proyectos y micro-obras de alto impacto social.


In [ ]:
print("--- Diagnóstico para ENTRE RÍOS ---")
diagnostico_er = diagnostico_fragmentacion_inversion(inversiones_df, 'ENTRE RÍOS', MEDIA_MONTO_NACIONAL, MEDIA_PROYECTOS_NACIONAL)
print(f"Conclusión: {diagnostico_er['Conclusion']}")
print(f"Recomendación: {diagnostico_er['Recomendacion']}")

--- Diagnóstico para ENTRE RÍOS ---
Conclusión: ALERTA: Inversión altamente fragmentada en proyectos de bajo monto. Posible ineficiencia administrativa.
Recomendación: Evaluar la consolidación de proyectos futuros para aumentar el impacto económico y reducir la carga de gestión.


## 5️⃣ Evaluación de Métodos


Para nuestro Sistema Experto de Monitoreo de Obras Públicas, utilizamos dos técnicas clave de representación de conocimiento. Primero, los Marcos (Frames) se implementaron para la representación descriptiva y modular de la información. Cada Frame actúa como un objeto, como `Proyecto_de_Infraestructura`, con slots o atributos bien definidos (`id_proyecto`, `monto_total`, `avance_fisico`). Esto fue esencial para estructurar los datos heterogéneos del dataset en una jerarquía lógica, facilitando que el motor de inferencia accediera a los "hechos" necesarios para la evaluación. Sin embargo, los Frames solo organizan; no contienen la lógica para generar nuevos diagnósticos por sí mismos.

En segundo lugar, y fundamentalmente, se utilizaron las Reglas SI-ENTONCES (Programación Lógica) para la representación operativa y de diagnóstico. Para hacer la lógica robusta, incorporamos un principio de Ciencia de Datos: la escalabilidad. Modificamos umbrales fijos, asegurando que el sistema sea justo y preciso para provincias con carteras de proyectos de diferentes tamaños (como Córdoba vs. Santa Cruz).

Las Reglas SI-ENTONCES resultaron ser la técnica más efectiva y el pilar de nuestro sistema. Al ser ejecutadas bajo una estrategia de Encadenamiento Hacia Adelante (Forward Chaining), logramos un flujo de datos que va desde los hechos iniciales hacia la generación  de conocimiento. El sistema no espera una pregunta; simplemente procesa la cartera de proyectos provincial y dispara todas las alertas que se cumplen automáticamente. Esta capacidad de generación de alertas y diagnósticos justificados es lo que le da valor al Sistema Experto, transformando la inmensa tabla de datos en informes directos, lo cual es fundamental para el monitoreo en tiempo real de la gestión de la Obra Pública.

## 6️⃣ Conclusión Final del Grupo


Este proyecto fue fundamental para entender cómo se aplica la Inteligencia Artificial. El principal conocimiento fue la capacidad de transformar la experiencia humana en código  mediante un Sistema Experto. Aprendimos a usar las Reglas SI-ENTONCES para codificar la lógica de un analista (el motor de inferencia) y el Encadenamiento Hacia Adelante para crear un sistema de monitoreo que va de los datos (hechos) a las conclusiones (diagnóstico). Lo más impactante fue ver cómo el sistema detectó automáticamente riesgos críticos como la Sobrecertificación y los Proyectos Zombie, que son difíciles de identificar solo mirando tablas grandes.

Con la práctica, pudimos entender mucho más que sólo basándonos en la teoría. Por nuestra parte, creemos que ahora sabemos mejor cómo es el funcionamiento de un Sistema Experto y qué tan importante puede ser para reconocer patrones y generar diagnósticos.

Dentro de las posibles mejoras, la ampliación más emocionante es la integración de Machine Learning. Por ejemplo: En lugar de solo diagnosticar un proyecto como zombie en el presente, podríamos entrenar un modelo de clasificación usando variables de inicio (monto, duración estimada, sector) para predecir si un proyecto tiene alta probabilidad de volverse zombie en los próximos seis a doce meses. Esto transformaría el sistema en una herramienta predictiva que permite la intervención temprana, lo cual es el siguiente gran paso en la gestión de infraestructuras basada en datos.